# Õppetund 11 - Agentidevaheline (A2A) protokoll


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mis on A2A protokoll?

The **Agent-to-Agent (A2A) protocol** on avatud standard, mis võimaldab AI-agentidel suhelda, üksteist leida ja koostööd teha — isegi kui need on ehitatud erinevatele raamistikutele või majutatud erinevate teenuste poolt.

Key concepts:

- **Discovery** – Agendid avaldavad *Agendi kaart*, mis kirjeldab nende võimekust, muutes teiste agentide (või orkestraatorite) jaoks sobiva spetsialisti leidmise lihtsaks.
- **Message Passing** – Agendid vahetavad struktureeritud sõnumeid ühise protokolli kaudu, nii et ühe agendi päringut saab mõista ja täita teine, sõltumata selle sisemisest rakendusest.
- **Task Lifecycle** – A2A määratleb olekud nagu *esitatud*, *töös*, *valmis* ja *ebaõnnestunud*, andes orkestraatorile täieliku ülevaate sellest, kuidas delegeeritud ülesanne edeneb.

Selles õppetükis simuleerime A2A-laadset koostööd, ühendades kolm spetsialiseerunud reisialast agenti töövoogu, kus iga agent panustab oma ekspertiisiga ja edastab tulemused järgmisele.


## Spetsialiseeritud reisiagentide loomine


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Mitmeagendi koostöö töövoo kaudu

Me ühendame need kolm agenti järjestikusesse töövoogu, mis peegeldab agentilt-agentile (A2A) sõnumivahetust:

1. **CurrencyExchangeAgent** võtab vastu kasutaja päringu ja annab valuutanõuandeid.
2. **ActivityPlannerAgent** võtab vastu rikastatud konteksti ja lisab tegevussoovitusi.
3. **TravelManagerAgent** sünteesib mõlemad sisendid lõplikuks reisikokkuvõtteks.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## A2A mõistmine tootmiskeskkonnas

Tootmiskeskkonnas avab A2A-protokoll võimsad teenustevahelised stsenaariumid:

| Capability | Description |
|---|---|
| **Raamistikeülene koostöö** | Ühe raamistikuga ehitatud agent saab delegeerida ülesandeid agentile, mis on ehitatud mõnele teisele A2A-ühilduvale raamistikule, võimaldades tõelist organisatsioonidevahelist ühilduvust. |
| **Teenusepiirid** | Agendid võivad asuda eraldi mikroteenustes, pilveregioonides või isegi erinevates organisatsioonides, kuid teha koostööd sujuvalt. |
| **Dünaamiline avastamine** | Orkestreerija saab käituseajal pärida Agent Card registrit, et leida antud alaülesande jaoks sobivaim spetsialist. |
| **Voogedastus & push-teavitused** | A2A toetab Server-Sent Events (SSE) reaalajas edenemise uuendusteks ja push-teavitusi pikaajaliste ülesannete jaoks. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## Kokkuvõte

Selles õppetükis õppisite:

1. **Mis on A2A-protokoll** — avatud standard agendi-agendi avastamiseks, sõnumivahetuseks,
   ja ülesannete haldamiseks.
2. **Kuidas luua spetsialiseeritud agente** — valuutavahetuse agent, tegevuste planeerija agent,
   ja reisihalduri orkestreerija.
3. **Kuidas ühendada agente töövoogu** — kasutades `WorkflowBuilder` järjestikulise
   sõnumiedastuse modelleerimiseks agentide vahel.
4. **Kuidas A2A töötab tootmises** — võimaldades raamistikevahelist, teenustevahelist koostööd
   dünaamilise avastamise ja voogedastuse värskendustega.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vastutusest loobumine:
See dokument on tõlgitud tehisintellekti tõlke­teenuse Co‑op Translator (https://github.com/Azure/co-op-translator) abil. Kuigi me püüame tagada tõlkete täpsust, palun arvestage, et automatiseeritud tõlked võivad sisaldada vigu või ebatäpsusi. Algdokumenti selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate arusaamatuste ega valesti tõlgenduste eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
